Currenyly we are managing the prompt by including the promp as part of the fn 
but its
- hard to version manage
- takes space
- is not centralised
- limits who can work with the prompts
    

we will look into 
- jinja templates
- prompt registires 

### Import Dependancies

In [12]:
import yaml
from jinja2 import Template 
from langsmith import Client

### RAG pipeline prompt

In [13]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt

f strings have their limitations
i.e if

In [14]:
preprocessed_context = "-a \n-b"
question = "what is a?"

In [15]:

prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

In [16]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
-a 
-b

Question:
what is a?
    


### Jinja Templates

this is a templating framework used for templating strings

In [29]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{preprocessed_context}}

Question:
{{question}}
"""

In [30]:
template = Template(jinja_template)

In [31]:
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [32]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
-a 
-b

Question:
what is a?


this is a normalisation of how to use templating 
they also allow for 
- advanced functionality such 
- properly injecting json variables
- combining multiple jinja templates
- conditional statements and loops in the template

In [33]:
from calendar import JUNE


def build_prompt_jinja(preprocessed_context, question):   

    jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{preprocessed_context}}

Question:
{{question}}
    """
    template = Template(jinja_template)
    rendered_template = template.render(
        preprocessed_context=preprocessed_context, 
        question=question
    )

    return rendered_template

In [34]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
-a 
-b

Question:
what is a?
    


In [35]:
print(build_prompt_jinja("- some item", "- some question"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- some item

Question:
- some question
    


we will use a configuration yaml file to store our prompts
- this will allow us to manage access
- and version control 

In [36]:
def prompt_template_config(yaml_path, prompt_key):

    with open(yaml_path, "r") as file:
        config = yaml.safe_load(file)
    
    template_content = config["prompts"][prompt_key]

    template = Template(template_content)

    return template

In [37]:
template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

In [38]:
template

<Template memory:113b5dbe0>

In [40]:
rendered_prompt = template.render(
    preprocessed_context=preprocessed_context,
    question=question
)

In [41]:
print(rendered_prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
-a 
-b

Question:
what is a?


we can also define things like the models in the prompts as well as other prompts when using model routing

In [42]:
def build_prompt_jinja(preprocessed_context, question):

    template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

    rendered_prompt = template.render(
    preprocessed_context=preprocessed_context,
    question=question
    )

    return rendered_prompt

In [43]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
-a 
-b

Question:
what is a?


### Prompt Registries

if you go to langsmith ui you can see the prompts tab - here we can build prompts

we have saved the system ptompt in langsmith an named it retrieval-generation

In [44]:
ls_client = Client()

In [47]:
ls_template = ls_client.pull_prompt("retrieval-generation ")

LangSmithError: Failed to GET /commits/-/retrieval-generation /latest in LangSmith API. HTTPError('400 Client Error: Bad Request for url: https://api.smith.langchain.com/commits/-/retrieval-generation%20/latest', '{"error":"No prompt owner specified"}\n')

In [48]:
ls_template

NameError: name 'ls_template' is not defined

In [49]:
print(ls_template.messages[0].prompt.template)

NameError: name 'ls_template' is not defined

In [50]:
def promp_template_registry(prompt_name):

    template_content = ls_client.pull_prompt(prompt_name).message[0].prompt.template

    template = Template(template_content)

    return template

In [51]:
print(
    promp_template_registry("retrieval_generation").render(
        preprocessed_context=preprocessed_context,
        question=question
    )
)

LangSmithError: Failed to GET /commits/-/retrieval_generation/latest in LangSmith API. HTTPError('400 Client Error: Bad Request for url: https://api.smith.langchain.com/commits/-/retrieval_generation/latest', '{"error":"No prompt owner specified"}\n')

if only engineers need access to the prompt -> yaml
if SME's need access -> langsmith (promt registry) outside of codebase

version management and collaboration 

now we will add the yaml to the backend

will crate a utils script to parse the prompt

`uv add --package api pyyaml`